<a href="https://colab.research.google.com/github/nagahemanth68-crypto/NLP/blob/main/NLP_Task_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import re
import requests
import torch
import torch.nn as nn
from bs4 import BeautifulSoup
from tensorflow.keras.utils import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer


class ChunkerModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 128)
        self.lstm = nn.LSTM(128, 64, batch_first=True)
        self.fc = nn.Linear(64, 1)

    def forward(self, x):
        x = self.embedding(x)
        return torch.sigmoid(self.fc(self.lstm(x)[0][:, -1, :]))


def fetch_text(url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"
    }
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    text = " ".join(p.get_text() for p in soup.find_all("p"))
    return re.sub(r"\s+", " ", text).strip()


def preprocess(text):
    tok = Tokenizer(num_words=5000)
    tok.fit_on_texts([text])
    seq = tok.texts_to_sequences([text])
    return pad_sequences(
        seq, maxlen=100, padding="post"
    ), tok


def segment_text(url):
    text = fetch_text(url)
    seq, tok = preprocess(text)
    return text.split(". ")[:5]


url = "https://en.wikipedia.org/wiki/Natural_language_processing"

print("Extracted Chunks:", *segment_text(url), sep="\n")

Extracted Chunks:
Natural language processing (NLP) is the processing of natural language information by a computer
NLP is a subfield of computer science and is closely associated with artificial intelligence
NLP is also related to information retrieval, knowledge representation, computational linguistics, and linguistics more broadly.[1] Major processing tasks in an NLP system include: speech recognition, text classification, natural language understanding, and natural language generation
Natural language processing has its roots in the 1950s.[2] Already in 1950, Alan Turing published an article titled "Computing Machinery and Intelligence," which proposed what is now called the Turing test as a criterion of intelligence, though at the time that was not articulated as a problem separate from artificial intelligence
The proposed test includes a task that involves the automated interpretation and generation of natural language
